In [2]:
import pandas as pd
import numpy as np
import holidays

In [3]:
df = pd.read_csv("Utility_consumption_cleaned.csv", parse_dates=["Datetime"])
df = df.set_index("Datetime").sort_index()

targets = ["Total_PowerConsumption", "F1_132KV_PowerConsumption",
           "F2_132KV_PowerConsumption", "F3_132KV_PowerConsumption"]

df.head()

,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,Total_PowerConsumption
Datetime,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,70425.53544
2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,69320.84387
2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,67803.22193
2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,65489.23209
2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,63650.44627


### Feature Engineering Setup — Notes

Loading the cleaned dataset and defining all four prediction targets upfront 
(`Total_PowerConsumption`, `F1`, `F2`, `F3`). Every feature built in this section will be 
shared across all four targets, except the lag/rolling features, which are generated 
per-target — since the EDA established that each feeder has a distinct demand profile and 
needs its own "memory" of recent behavior.

# Calendar / Time Features

In [4]:
df["hour"] = df.index.hour
df["minute"] = df.index.minute
df["time_of_day_frac"] = df["hour"] + df["minute"] / 60.0   # continuous 0-24
df["dayofweek"] = df.index.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["is_sunday"] = (df["dayofweek"] == 6).astype(int)
df["month"] = df.index.month
df["dayofyear"] = df.index.dayofyear
df["block_of_day"] = df["hour"] * 6 + df["minute"] // 10     # 0-143, matches 10-min block structure

df[["hour", "dayofweek", "is_weekend", "is_sunday", "month", "block_of_day"]].head()

,hour,dayofweek,is_weekend,is_sunday,month,block_of_day
Datetime,,,,,,
2017-01-01 00:00:00,0,6,1,1,1,0
2017-01-01 00:10:00,0,6,1,1,1,1
2017-01-01 00:20:00,0,6,1,1,1,2
2017-01-01 00:30:00,0,6,1,1,1,3
2017-01-01 00:40:00,0,6,1,1,1,4


In [5]:
df.head()

,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,Total_PowerConsumption,hour,minute,time_of_day_frac,dayofweek,is_weekend,is_sunday,month,dayofyear,block_of_day
Datetime,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,70425.53544,0,0,0.000000,6,1,1,1,1,0
2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,69320.84387,0,10,0.166667,6,1,1,1,1,1
2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,67803.22193,0,20,0.333333,6,1,1,1,1,2
2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,65489.23209,0,30,0.500000,6,1,1,1,1,3
2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,63650.44627,0,40,0.666667,6,1,1,1,1,4


### Calendar / Time Features — Justification

These features are built directly from EDA findings:
- `hour` / `block_of_day`: the single-day zoom plots and ACF/PACF both confirmed an 
  extremely strong, repeating 24-hour cycle — time-of-day is one of the most informative 
  signals available.
- `dayofweek` / `is_sunday` (rather than a generic `is_weekend`): the day-of-week analysis 
  showed Sunday specifically as the lower day, while Saturday behaved like a normal 
  weekday — a plain weekend flag would misrepresent this. `is_weekend` is still included 
  for completeness/comparison, but `is_sunday` is expected to carry more signal.
- `month` / `dayofyear`: needed to let the model separate seasonal effects (e.g., summer 
  cooling load) from daily/weekly cycles.

# Cyclical Encodings (sin/cos)

In [6]:
# Sin/cos encoding avoids an artificial jump at the hour=23 -> hour=0 boundary
df["hour_sin"] = np.sin(2*np.pi*df["time_of_day_frac"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["time_of_day_frac"]/24)
df["dow_sin"] = np.sin(2*np.pi*df["dayofweek"]/7)
df["dow_cos"] = np.cos(2*np.pi*df["dayofweek"]/7)
df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

df[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].head()

,hour_sin,hour_cos,dow_sin,dow_cos
Datetime,,,,
2017-01-01 00:00:00,0.000000,1.000000,-0.781831,0.62349
2017-01-01 00:10:00,0.043619,0.999048,-0.781831,0.62349
2017-01-01 00:20:00,0.087156,0.996195,-0.781831,0.62349
2017-01-01 00:30:00,0.130526,0.991445,-0.781831,0.62349
2017-01-01 00:40:00,0.173648,0.984808,-0.781831,0.62349


In [7]:
df.head()

,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,Total_PowerConsumption,hour,minute,...,is_sunday,month,dayofyear,block_of_day,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,70425.53544,0,0,...,1,1,1,0,0.000000,1.000000,-0.781831,0.62349,0.5,0.866025
2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,69320.84387,0,10,...,1,1,1,1,0.043619,0.999048,-0.781831,0.62349,0.5,0.866025
2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,67803.22193,0,20,...,1,1,1,2,0.087156,0.996195,-0.781831,0.62349,0.5,0.866025
2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,65489.23209,0,30,...,1,1,1,3,0.130526,0.991445,-0.781831,0.62349,0.5,0.866025
2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,63650.44627,0,40,...,1,1,1,4,0.173648,0.984808,-0.781831,0.62349,0.5,0.866025


### Cyclical Encoding — Justification

Raw `hour` (0-23) and `dayofweek` (0-6) treat the boundary between the last and first value 
(e.g., hour 23 -> hour 0) as a large jump, even though they're adjacent in real time. Since 
the daily cycle is smooth and continuous (confirmed by the seasonal decomposition and the 
single-day load curve), sin/cos encoding represents time-of-day and day-of-week as points 
on a circle, preserving their true continuity. This is especially relevant for any 
linear/distance-based model and is harmless to include even for tree-based models that 
don't strictly need it.

# Weather Feature Transforms

In [8]:
# WindSpeed -> near-binary fan/blower status (per EDA bimodality finding)
df["is_fan_on"] = (df["WindSpeed"] > 2.5).astype(int)

# Temperature: non-linear cooling-load feature (per LOWESS finding — steep rise above ~22C)
df["cooling_degree"] = np.clip(df["Temperature"] - 22, a_min=0, a_max=None)
df["temp_sq"] = df["Temperature"] ** 2

# CloudCover: retain the raw candidate and expose its temperature-dependent signal
# (EDA: modest target correlation, but substantial shared seasonality with Temperature)
df["cloud_temp_interaction"] = df["CloudCover"] * df["Temperature"]

df[["WindSpeed", "is_fan_on", "Temperature", "cooling_degree", "temp_sq",
    "CloudCover", "cloud_temp_interaction"]].head()

,WindSpeed,is_fan_on,Temperature,cooling_degree,temp_sq,CloudCover,cloud_temp_interaction
Datetime,,,,,,,
2017-01-01 00:00:00,0.083,0,6.559,0.0,43.020481,18.000000,118.062000
2017-01-01 00:10:00,0.083,0,6.414,0.0,41.139396,18.833333,120.797000
2017-01-01 00:20:00,0.080,0,6.313,0.0,39.853969,19.666667,124.155667
2017-01-01 00:30:00,0.083,0,6.121,0.0,37.466641,20.500000,125.480500
2017-01-01 00:40:00,0.081,0,5.921,0.0,35.058241,21.333333,126.314667


### Weather Feature Transforms — Justification

- `is_fan_on`: directly addresses the WindSpeed anomaly found in EDA — values clustered 
  almost entirely into two narrow bands (~0.05-0.09 and ~4.9-4.93) with a block-like 
  switching pattern over time, far more consistent with a cooling fan/blower on/off status 
  than continuous wind measurement. Converting it to a binary feature respects what the 
  data actually represents rather than treating it as a continuous weather variable, which 
  could mislead a model into learning spurious fine-grained relationships.
- `cooling_degree` (max(0, Temperature - 22)): directly encodes the non-linear threshold 
  behavior found in the LOWESS curve, where load rises gently at lower temperatures but 
  much more steeply above ~22°C. This gives the model an explicit cooling-load signal 
  instead of relying on it to discover the threshold purely from raw Temperature.
- `temp_sq`: a simpler, complementary non-linear transform included for comparison against 
  `cooling_degree` during feature selection / model training.
- `CloudCover` is retained as a raw candidate because EDA found a modest positive association 
  with total demand (~0.30, and ~0.34 for F3). Its correlation with Temperature (~0.60) shows 
  that much of this may be shared seasonality, so it should not be interpreted as an independent 
  causal driver. `cloud_temp_interaction` makes a conditional cloud/temperature effect available 
  to linear models; tree-based models may learn the interaction from the raw inputs directly. 
  Both cloud features must be retained only if time-based ablation shows incremental validation 
  value after calendar, temperature, and holiday features are already present.

# Holiday Features (2017 Calendar)

In [9]:
jh_2017 = holidays.India(years=2017, subdiv='JH', categories=('public', 'optional'))
lib_2017 = {pd.to_datetime(d): name for d, name in jh_2017.items()}

manual_holidays_2017 = {
    "2017-03-30": "Sarhul", "2017-09-04": "Karma Puja", "2017-10-20": "Sohrai",
    "2017-01-16": "Tusu Parab", "2017-06-30": "Hul Diwas", "2017-09-17": "Vishwakarma Puja",
    "2017-08-20": "Mansha Puja", "2017-02-11": "Mage Parab", "2017-06-25": "Rath Yatra",
    "2017-08-25": "Teej Parab", "2017-09-13": "Jivitputrika Parab", "2017-02-01": "Basant Panchami"
}
manual_2017 = {pd.to_datetime(d): name for d, name in manual_holidays_2017.items()}
all_holidays_2017 = {**lib_2017, **manual_2017}
holiday_dates = set(pd.to_datetime(list(all_holidays_2017.keys())).normalize())

df["date"] = df.index.normalize()
df["is_holiday"] = df["date"].isin(holiday_dates).astype(int)
df["is_pre_holiday"] = df["date"].apply(lambda d: (d + pd.Timedelta(days=1)) in holiday_dates).astype(int)
df["is_post_holiday"] = df["date"].apply(lambda d: (d - pd.Timedelta(days=1)) in holiday_dates).astype(int)

holiday_arr = pd.to_datetime(sorted(holiday_dates))
unique_dates = df["date"].unique()
nearest_map = {d: np.min(np.abs((holiday_arr - d).days)) for d in unique_dates}
df["days_to_nearest_holiday"] = df["date"].map(nearest_map)

df[["date", "is_holiday", "is_pre_holiday", "is_post_holiday", "days_to_nearest_holiday"]].drop_duplicates("date").head(10)

,date,is_holiday,is_pre_holiday,is_post_holiday,days_to_nearest_holiday
Datetime,,,,,
2017-01-01,2017-01-01,0,0,0,13
2017-01-02,2017-01-02,0,0,0,12
2017-01-03,2017-01-03,0,0,0,11
2017-01-04,2017-01-04,0,0,0,10
2017-01-05,2017-01-05,0,0,0,9
2017-01-06,2017-01-06,0,0,0,8
2017-01-07,2017-01-07,0,0,0,7
2017-01-08,2017-01-08,0,0,0,6
2017-01-09,2017-01-09,0,0,0,5


### Holiday Features — Justification

Reuses the localized 2017 Dhanbad/Jharkhand holiday calendar built earlier (library-computed 
lunar dates + manually sourced tribal/regional festivals from the BBMKU Dhanbad University 
list). 

Importantly, the holiday EDA found that `is_holiday` alone is a **weak and potentially 
misleading** signal when used in isolation — several holidays showed large apparent 
increases in consumption that were actually driven by seasonal/temperature effects, not the 
holiday itself (e.g., August holidays coinciding with peak cooling season). `is_holiday` is 
therefore included **alongside** `cooling_degree`, `month`, and the cyclical month 
encoding — never as a standalone feature — so the model has the context needed to 
disentangle the true calendar effect from confounded seasonal effects. `is_pre_holiday`, 
`is_post_holiday`, and `days_to_nearest_holiday` are included as secondary features, 
consistent with the EDA finding of a small, gradual (not sharp) pre/post-holiday effect.

# Lag Features (per target)

In [10]:
# Lags chosen based on ACF/PACF findings: lag-1 and lag-144 are the strongest predictors
lag_steps = {
    "lag_1": 1,        # 10 minutes ago
    "lag_6": 6,         # 1 hour ago
    "lag_144": 144,      # 24 hours ago (same time yesterday) -- strongest seasonal lag
    "lag_288": 288,      # 48 hours ago
    "lag_1008": 1008,    # 7 days ago (same time last week)
}

for tgt in targets:
    for lag_name, lag_val in lag_steps.items():
        df[f"{tgt}_{lag_name}"] = df[tgt].shift(lag_val)

df.filter(like="Total_PowerConsumption_lag").head(10)

,Total_PowerConsumption_lag_1,Total_PowerConsumption_lag_6,Total_PowerConsumption_lag_144,Total_PowerConsumption_lag_288,Total_PowerConsumption_lag_1008
Datetime,,,,,
2017-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:10:00,70425.53544,NaN,NaN,NaN,NaN
2017-01-01 00:20:00,69320.84387,NaN,NaN,NaN,NaN
2017-01-01 00:30:00,67803.22193,NaN,NaN,NaN,NaN
2017-01-01 00:40:00,65489.23209,NaN,NaN,NaN,NaN
2017-01-01 00:50:00,63650.44627,NaN,NaN,NaN,NaN
2017-01-01 01:00:00,62171.34398,70425.53544,NaN,NaN,NaN
2017-01-01 01:10:00,60937.36065,69320.84387,NaN,NaN,NaN
2017-01-01 01:20:00,59566.75124,67803.22193,NaN,NaN,NaN


### Lag Features — Justification

Lag windows were chosen directly from the lag-autocorrelation and ACF/PACF analysis, not 
arbitrarily:
- `lag_1` (10 min): near-perfect correlation (0.997) — the single strongest predictor 
  available, capturing immediate momentum.
- `lag_144` (24h, same time yesterday): the PACF's second-strongest spike after lag-1, 
  correlation 0.979 — captures the daily seasonal cycle directly.
- `lag_288` (48h) and `lag_1008` (7 days): included to give the model context across 
  multiple days and a same-time-last-week comparison, both retaining correlations above 
  0.96.
- `lag_6` (1h): a shorter-horizon lag to help capture intra-hour momentum/trend that lag-1 
  alone might not fully represent.

These are generated **per target** (Total, F1, F2, F3 each get their own lag set) rather 
than only on Total, since the EDA showed each feeder has a distinct enough profile that 
its own recent history is more informative than a shared/aggregate lag would be.

# Rolling Window Features (per target)

In [11]:
roll_windows = {"roll_1h": 6, "roll_6h": 36, "roll_24h": 144}

for tgt in targets:
    for roll_name, window in roll_windows.items():
        # shift(1) BEFORE rolling -> window never includes the current row -> avoids data leakage
        df[f"{tgt}_{roll_name}_mean"] = df[tgt].shift(1).rolling(window).mean()
        df[f"{tgt}_{roll_name}_std"] = df[tgt].shift(1).rolling(window).std()

df.filter(like="Total_PowerConsumption_roll").head(10)

,Total_PowerConsumption_roll_1h_mean,Total_PowerConsumption_roll_1h_std,Total_PowerConsumption_roll_6h_mean,Total_PowerConsumption_roll_6h_std,Total_PowerConsumption_roll_24h_mean,Total_PowerConsumption_roll_24h_std
Datetime,,,,,,
2017-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:50:00,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 01:00:00,66476.770597,3253.951738,NaN,NaN,NaN,NaN
2017-01-01 01:10:00,64895.408132,3256.657331,NaN,NaN,NaN,NaN
2017-01-01 01:20:00,63269.726027,3032.561090,NaN,NaN,NaN,NaN


In [12]:
df.head()

,Temperature,Humidity,WindSpeed,F1_132KV_PowerConsumption,F2_132KV_PowerConsumption,F3_132KV_PowerConsumption,CloudCover,Total_PowerConsumption,hour,minute,...,F2_132KV_PowerConsumption_roll_6h_mean,F2_132KV_PowerConsumption_roll_6h_std,F2_132KV_PowerConsumption_roll_24h_mean,F2_132KV_PowerConsumption_roll_24h_std,F3_132KV_PowerConsumption_roll_1h_mean,F3_132KV_PowerConsumption_roll_1h_std,F3_132KV_PowerConsumption_roll_6h_mean,F3_132KV_PowerConsumption_roll_6h_std,F3_132KV_PowerConsumption_roll_24h_mean,F3_132KV_PowerConsumption_roll_24h_std
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,34055.69620,16128.87538,20240.96386,18.000000,70425.53544,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:10:00,6.414,74.5,0.083,29814.68354,19375.07599,20131.08434,18.833333,69320.84387,0,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:20:00,6.313,74.5,0.080,29128.10127,19006.68693,19668.43373,19.666667,67803.22193,0,20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:30:00,6.121,75.0,0.083,28228.86076,18361.09422,18899.27711,20.500000,65489.23209,0,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01-01 00:40:00,5.921,75.7,0.081,27335.69620,17872.34043,18442.40964,21.333333,63650.44627,0,40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Rolling Window Features — Justification

Rolling mean/std over 1h, 6h, and 24h windows capture local trend and volatility that a 
single lag point can't — e.g., whether load has been climbing steadily or fluctuating 
sharply in the recent past, which is informative beyond just "what was the value N steps 
ago." 

**Critical implementation detail:** each target is `.shift(1)` before the rolling window is 
applied. This ensures the rolling window only uses *past* values relative to the row being 
predicted, never the current target value itself — preventing data leakage that would 
otherwise make the model look artificially accurate during training but fail in real 
deployment, where the current value is exactly what's being predicted and can't be an input.

# Drop Warm-Up NaN Rows & Save

In [13]:
before = len(df)
df_features = df.dropna().reset_index()
after = len(df_features)

print(f"Rows before dropna: {before}")
print(f"Rows after dropna: {after}")
print(f"Dropped (warm-up period for the largest lag, 1008): {before - after}")

df_features.to_csv("Utility_features_engineered.csv", index=False)
print("Saved feature matrix shape:", df_features.shape)

Rows before dropna: 52416
Rows after dropna: 51408
Dropped (warm-up period for the largest lag, 1008): 1008
Saved feature matrix shape: (51408, 77)


### Handling Warm-Up NaNs — Justification

The largest lag (1008 steps = 7 days) means the first 1,008 rows of the dataset can't have 
a valid `lag_1008` value, since there's no data 7 days before the start of the series. 
These rows are dropped rather than imputed, since imputing fabricated values for the 
strongest engineered features would introduce artificial noise into exactly the part of 
the feature set the model relies on most. This costs only 1,008 of 52,416 rows (~1.9%) — 
an acceptable trade-off for feature integrity given the size of the remaining dataset 
(51,408 rows, still well over a full year of 10-minute data).

# Sanity Check: Feature Correlation with Target

In [14]:
target = "Total_PowerConsumption"
feature_cols = (
    [c for c in df_features.columns if c.startswith(f"{target}_")]
    + ["cooling_degree", "CloudCover", "cloud_temp_interaction", "hour_sin",
       "is_holiday", "is_pre_holiday", "is_post_holiday",
       "days_to_nearest_holiday"]
)

corr = df_features[feature_cols + [target]].corr()[target].sort_values(ascending=False)
print(corr)

Total_PowerConsumption                  1.000000
Total_PowerConsumption_lag_1            0.997033
Total_PowerConsumption_lag_144          0.979206
Total_PowerConsumption_roll_1h_mean     0.972502
Total_PowerConsumption_lag_288          0.970000
Total_PowerConsumption_lag_1008         0.965584
Total_PowerConsumption_lag_6            0.923587
Total_PowerConsumption_roll_6h_mean     0.657448
Total_PowerConsumption_roll_24h_mean    0.478392
cooling_degree                          0.450329
cloud_temp_interaction                  0.403978
Total_PowerConsumption_roll_24h_std     0.359754
CloudCover                              0.300339
Total_PowerConsumption_roll_1h_std      0.218219
days_to_nearest_holiday                 0.158815
Total_PowerConsumption_roll_6h_std      0.099808
is_pre_holiday                         -0.031927
is_holiday                             -0.032588
is_post_holiday                        -0.034455
hour_sin                               -0.711423
Name: Total_PowerCon

### Feature Sanity Check — Findings

The correlation check confirms the EDA's predictions hold up in the engineered feature set:
- `lag_1` (0.997) and `lag_144` (0.979) remain the dominant predictors, exactly as the 
  ACF/PACF analysis indicated.
- `roll_1h_mean` (0.973) performs nearly as well as the raw lags, suggesting it may 
  partially substitute for or complement individual lag features — worth examining via 
  feature importance once a model is trained.
- `cooling_degree` (0.450) shows a meaningful standalone correlation, validating the 
  non-linear temperature transform as more useful than relying on raw Temperature alone.
- `CloudCover` is included in this check as a candidate with a modest raw association. Because 
  it shares substantial seasonality with Temperature, its raw correlation is descriptive rather 
  than proof of incremental value; compare models with and without `CloudCover` and 
  `cloud_temp_interaction` using chronological validation.
- `is_holiday` shows a near-zero/slightly negative simple correlation (-0.033) **in 
  isolation** — this is expected and consistent with the earlier finding that holiday 
  effects are only visible once seasonal/temperature confounds are controlled for. A weak 
  standalone correlation here does **not** mean the feature is useless; it means its value 
  will only show up in combination with other features, which a multivariate model (unlike 
  a simple pairwise correlation) is able to capture.

This sanity check is a good checkpoint before model training: the strongest engineered 
features behave exactly as the EDA evidence predicted, giving confidence that the feature 
set is grounded in real patterns rather than guesswork.

## Summary: Feature Engineering

This feature set was built to directly operationalize every major finding from the EDA, 
rather than applying a generic feature template. The features span five categories:

**Calendar/cyclical features** (`hour`, `block_of_day`, `dayofweek`, `is_sunday`, `month`, 
sin/cos encodings) encode the extremely strong and smooth daily/weekly periodicity 
confirmed by the seasonal decomposition and ACF/PACF analysis, with `is_sunday` 
specifically chosen over a generic weekend flag based on the day-of-week finding that only 
Sunday — not Saturday — behaves as a lower-demand day.

**Weather features and transforms** (`CloudCover`, `cloud_temp_interaction`, `is_fan_on`, 
`cooling_degree`, `temp_sq`) address three distinct EDA 
findings: WindSpeed's bimodal, block-switching distribution (treated as a near-binary 
fan/blower status rather than continuous wind data) and Temperature's non-linear, 
threshold-like relationship with load (a steep rise above ~22°C consistent with cooling 
demand, with no symmetric heating-side effect in the observed range). Raw CloudCover is 
retained because of its modest demand association, while an interaction with Temperature 
allows its conditional contribution to be tested without mistaking shared seasonality for 
independent signal.

**Holiday features** (`is_holiday`, `is_pre_holiday`, `is_post_holiday`, 
`days_to_nearest_holiday`) reuse the localized Dhanbad/Jharkhand holiday calendar, but are 
deliberately included **alongside** seasonal/temperature features rather than in isolation 
— the holiday EDA specifically uncovered that several holidays' apparent effects were 
confounded with seasonal cooling load, so the holiday signal is only reliable once the 
model can separate it from co-occurring seasonal effects.

**Lag and rolling features**, generated independently for all four targets (Total, F1, F2, 
F3), are the backbone of the feature set — `lag_1` and `lag_144` alone retain correlations 
above 0.97 with their respective targets, directly reflecting the ACF/PACF evidence that 
these two lags capture the bulk of the predictable signal. Per-target (rather than 
shared/aggregate) lag features were chosen specifically because the EDA showed F1, F2, and 
F3 have distinct enough profiles that each feeder's own recent history is more informative 
than a pooled signal.

A leakage-safe design choice — shifting each target by 1 step before computing rolling 
statistics — ensures rolling mean/std features never have access to the value they're 
meant to help predict, which matters for producing a model that will generalize to real 
forecasting rather than appear artificially accurate during training.

The final sanity check confirmed that engineered feature correlations align with EDA 
expectations point-for-point: dominant lags, a meaningful cooling-degree signal, candidate 
cloud-cover signal, and a holiday flag that appears weak in isolation but may contribute once 
combined with other features in a multivariate model.

**Carried forward into Model Architecture:**
- The strength of `lag_1`/`lag_144` suggests either a recursive single-step model or a 
  direct multi-horizon model could work well — this trade-off should be evaluated 
  explicitly before committing to one in the next section.
- Per-feeder feature sets support training **four independent models**, consistent with 
  every prior EDA section's evidence of feeder-specific behavior.
- `CloudCover`, `cloud_temp_interaction`, and the holiday features should be evaluated with 
  chronological ablation (baseline vs. +cloud vs. +holidays vs. +both), then interpreted with 
  feature importance. This tests their incremental contribution beyond temperature and 
  calendar seasonality without relying on confounded pairwise correlations.